# 07e — Causal Example (DML)

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
from scorelab import tuning as T, explain as E, survival as SV, causal as CA
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

A short, **illustrative** causal estimate: the difference in the probability of going bad within 24 months between a 60-month and a 36-month term, for applicants with the same application attributes, by double machine learning (cross-fitted partially linear model, and the doubly robust interactive model for the ATE and ATT).

**This is not a causal claim for policy.** The borrower chooses the term and the lender prices it, so the 60-month group differs in ways no application attribute records (payment pressure, purpose, expected income), and the estimate cannot rule that out. Interest rate, grade and instalment are left out of the controls on purpose: the lender sets them after the term is chosen, so controlling for them would remove part of the effect being estimated.

Decision: **AC — the control set, the estimand, and how the number may be used.**

In [ ]:
CAUSAL_SAMPLE = 0.25                         # share of 2012-2016 loans (DML fits 3 learners x 5 folds)
cands = json.loads((MODELS / "candidate_attributes.json").read_text(encoding="utf-8"))["candidates"]
cf = A.build_attributes(CA.causal_frame(con, sample_frac=SAMPLE_FRAC or CAUSAL_SAMPLE))
print(f"{len(cf):,} loans · 60-month share {cf.treated.mean():.1%}")
display(cf.groupby(["vintage_year", "term_months"])["bad"].agg(["size", "mean"]).unstack().round(4))

### Decision AC — Controls and estimand

Controls: the bureau candidates plus application fields (amount, income, DTI, employment length, purpose, home ownership). Is amount a confounder (it drives both term choice and risk) or partly a consequence of the term (a longer term makes a larger amount affordable)? ATE (every applicant) or ATT (applicants who chose 60 months)? What may the number be used for: a sanity check on the size of the term effect in the risk ranking, never a pricing or product decision.

**Your reasoning:**

_(write here)_

In [ ]:
controls = [c for c in cands if c in cf] + CA.APPLICATION_CONTROLS
X = CA.control_matrix(cf, controls, CA.APPLICATION_CATEGORICALS)
y, d = cf["bad"].to_numpy(), cf["treated"].to_numpy()
bench = CA.naive_and_adjusted(y, d, X)
plr = CA.dml_plr(y, d, X, n_folds=5)
irm = CA.dml_irm(y, d, X, n_folds=5, trim=0.01)
est = pd.concat([bench, pd.DataFrame([
    {"estimator": "DML partially linear (theta)", "effect": plr["theta"], "ci_low": plr["ci"][0], "ci_high": plr["ci"][1]},
    {"estimator": "DML interactive, ATE", "effect": irm["ate"], "ci_low": irm["ate_ci"][0], "ci_high": irm["ate_ci"][1]},
    {"estimator": "DML interactive, ATT", "effect": irm["att"], "ci_low": irm["att_ci"][0], "ci_high": irm["att_ci"][1]}])], ignore_index=True)
display(est.round(4))
print(f"nuisance fits: outcome AUC {plr['outcome_auc']:.3f} · propensity AUC {plr['propensity_auc']:.3f} · trimmed {irm['n_propensity_trimmed']:,} of {irm['n']:,}")
est.to_csv(config.DOCS_DIR / "exhibits" / "dml_term_effect.csv", index=False)

In [ ]:
# Evidence — overlap: are there 36- and 60-month borrowers with the same propensity?
ov = CA.overlap_table(irm["propensity"], d); display(ov)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(irm["propensity"][d == 0], bins=50, alpha=0.6, density=True, label="36 months")
ax.hist(irm["propensity"][d == 1], bins=50, alpha=0.6, density=True, label="60 months")
ax.set_xlabel("estimated propensity of a 60-month term"); ax.legend(); plt.show()

In [ ]:
# Evidence — is the estimate stable across vintages? (partially linear, 3 folds per year)
rows = []
for yr, g in cf.groupby("vintage_year"):
    Xg = CA.control_matrix(g, controls, CA.APPLICATION_CATEGORICALS)
    r = CA.dml_plr(g["bad"].to_numpy(), g["treated"].to_numpy(), Xg, n_folds=3)
    rows.append({"vintage_year": yr, "n": r["n"], "theta": r["theta"], "ci_low": r["ci"][0], "ci_high": r["ci"][1]})
display(pd.DataFrame(rows).round(4))
print(CA.CAVEAT)

### Interpretation

Write the one-paragraph reading of the numbers above, beginning with the caveat: what the adjusted difference is, how far it moved from the raw difference, whether it is stable across vintages, and why it is not the effect of a policy change.

**Your text:**

_(write here)_